# Phase 6: Dwell-Time Events Exploration

This notebook explores the stateful event lifecycle state machine (`candidate -> active -> triggered -> resolved`) per `(track_id, zone_id)`:
- Evaluating `events.csv` (`zone_entry`, `zone_exit`, `dwell_time_event`, `line_crossing`) on `people-detection.mp4`.
- Verifying hysteresis / grace-period (`grace_frames=5`) absorption on raw (`w=1`) boundary-flicker trajectories.
- Inspecting `event_debug_log/stay_traces.json` and `track_lost_in_zone` resolutions.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while (
    not (PROJECT_ROOT / "pyproject.toml").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "cv_projects" / "06_dwell_time_events" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from generate_dwell_events import (
    evaluate_dwell_events,
    load_events_config,
    load_scene_config,
    read_trajectories_csv,
)

In [ ]:
traj_csv = (
    PROJECT_ROOT
    / "cv_projects"
    / "04_trajectory_extraction"
    / "outputs"
    / "people_detection"
    / "trajectories.csv"
)
scene_cfg_path = (
    PROJECT_ROOT
    / "cv_projects"
    / "05_zone_line_reasoning"
    / "configs"
    / "scene_config.yaml"
)
events_cfg_path = (
    PROJECT_ROOT / "cv_projects" / "06_dwell_time_events" / "configs" / "events.yaml"
)
if traj_csv.exists() and scene_cfg_path.exists() and events_cfg_path.exists():
    trajs = read_trajectories_csv(traj_csv)
    scene_cfg = load_scene_config(scene_cfg_path)
    ev_cfg = load_events_config(events_cfg_path)
    events, traces = evaluate_dwell_events(trajs, scene_cfg, ev_cfg)
    for ev in events:
        print(ev.to_row())
    print(json.dumps([tr.to_dict() for tr in traces], indent=2))